In [1]:
import os

from io import StringIO
from google.cloud import storage
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
from peft import get_peft_model, LoraConfig, TaskType


/opt/anaconda3/envs/MachLearn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from plm_compare_progen2 import *
from plm_compare_esm import *
from protein_data import *
from pro_gen2_lora import *

In [3]:
load_dotenv()
cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

client = storage.Client()
bucket = client.bucket('domainome-data')
blob = bucket.blob('SupplementaryTable2.txt')

df = pd.read_csv(StringIO(blob.download_as_text()), sep='\t')

In [7]:
df.head()

,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank,pfam_ID
0,A0A2R8Y422_PF00240_2,A0A2R8Y422,*IFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,*,True,118.0,113.0,62.0,10.0,29.0,2.0,97.66667,0.030945,0.014885,-0.819050,0.208478,339,PF00240
1,A0A2R8Y422_PF00240_2,A0A2R8Y422,AIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,A,False,219.0,277.0,217.0,86.0,225.0,137.0,237.66670,0.069376,0.006673,-0.280790,0.093461,339,PF00240
2,A0A2R8Y422_PF00240_2,A0A2R8Y422,CIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,C,False,706.0,726.0,459.0,768.0,507.0,616.0,630.33330,0.082052,0.004141,-0.103250,0.057995,339,PF00240
3,A0A2R8Y422_PF00240_2,A0A2R8Y422,DIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,D,False,407.0,431.0,323.0,508.0,159.0,111.0,387.00000,0.071003,0.005162,-0.258003,0.072296,339,PF00240
4,A0A2R8Y422_PF00240_2,A0A2R8Y422,EIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,E,False,37.0,56.0,37.0,201.0,102.0,95.0,43.33333,0.116326,0.012085,0.376783,0.169263,339,PF00240


In [22]:
df[df['pfam_ID']=='O14901']

,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank,pfam_ID


In [23]:
pfam_dict={}

In [36]:
for x in df['pfam_ID'].unique():
    print(x)
    uid_list = list(df[df['pfam_ID']==x]['uniprot_ID'])
    uid_list = list(set(uid_list))
    print(uid_list)
    pfam_dict[x] = dict.fromkeys(uid_list, None)

PF00240
['A0A2R8Y422']
PF00096
['Q9NSC2', 'Q9UQR1', 'O43167', 'O60281', 'Q9Y2G7', 'P51508', 'Q9HC78', 'P37275', 'P25490', 'Q9H9D4', 'Q9BRP0', 'P08151', 'O14901', 'Q9NU63', 'Q99684', 'Q8NEA6', 'A0PJY2', 'Q5VTD9', 'P57071', 'Q9NQX1', 'Q9Y462', 'Q9NQV8', 'P10071', 'Q8TDD2', 'P10070', 'Q9UJQ4', 'O60481', 'Q9HAZ2']
PF00018
['Q7Z6B7', 'P07751', 'Q9UNF0', 'A1X283', 'Q15811', 'O15259', 'P02549', 'P51451', 'Q06187', 'Q12929', 'P19878', 'Q6XZF7', 'P12931', 'P08631', 'P62993', 'P14598', 'P15498', 'Q12965', 'P20936', 'P06239', 'Q5HYK7', 'Q08881', 'Q9Y5K6', 'O43295', 'O43586', 'P06241', 'Q8N157', 'P16885', 'P00519']
PF01846
['O75400', 'Q6NWY9', 'A2RRE5', 'O14776', 'Q5VWI1']
PF07525
['A6NK59', 'Q9NYS7']
rockdoms
['EHEE-rd1-0882', 'HHH-rd1-0142', 'EEHEE-rd3-0037']
PF00397
['O95817', 'Q96J02', 'Q6ZWJ1', 'Q96PU5', 'Q9H0M0', 'Q13526', 'Q5TCQ9', 'Q86UL8', 'Q9BYW2', 'O75400', 'Q8N3X1', 'Q9P2P5', 'Q9HAU4', 'Q96QZ7', 'P39940', 'Q9GZV5', 'P46934', 'O00308', 'P46937']
PF02817
['O00330', 'P11961', 'P11182']
PF

{'PF00240': {'A0A2R8Y422': None},
 'PF00096': {'Q9NSC2': None,
  'Q9UQR1': None,
  'O43167': None,
  'O60281': None,
  'Q9Y2G7': None,
  'P51508': None,
  'Q9HC78': None,
  'P37275': None,
  'P25490': None,
  'Q9H9D4': None,
  'Q9BRP0': None,
  'P08151': None,
  'O14901': None,
  'Q9NU63': None,
  'Q99684': None,
  'Q8NEA6': None,
  'A0PJY2': None,
  'Q5VTD9': None,
  'P57071': None,
  'Q9NQX1': None,
  'Q9Y462': None,
  'Q9NQV8': None,
  'P10071': None,
  'Q8TDD2': None,
  'P10070': None,
  'Q9UJQ4': None,
  'O60481': None,
  'Q9HAZ2': None},
 'PF00018': {'Q7Z6B7': None,
  'P07751': None,
  'Q9UNF0': None,
  'A1X283': None,
  'Q15811': None,
  'O15259': None,
  'P02549': None,
  'P51451': None,
  'Q06187': None,
  'Q12929': None,
  'P19878': None,
  'Q6XZF7': None,
  'P12931': None,
  'P08631': None,
  'P62993': None,
  'P14598': None,
  'P15498': None,
  'Q12965': None,
  'P20936': None,
  'P06239': None,
  'Q5HYK7': None,
  'Q08881': None,
  'Q9Y5K6': None,
  'O43295': None,
  'O435

In [ ]:
for x in pfam_dict.keys():
    pfam_dict[x] = 

ValueError: dictionary update sequence element #0 has length 10; 2 is required

In [29]:
pfam_dict

{'PF00240': ['A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422',
  'A0A2R8Y422

In [15]:
domainome_ids = set(df['uniprot_ID'])

In [19]:
for x in domainome_ids:
    if 'A' in x:
        print(x)

A2RRE5
Q9C0A1
A0PJY2
Q9H0A6
A6NK59
Q96AE4
Q9H4A3
Q8TAQ2
Q9HA38
Q9UHA3
A1X283
P0A9X9
A0A2R8Y422
Q9HAU4
Q8NEA6
Q8TAT5
Q9HAZ2


In [ ]:
# # Load environment variables
# load_dotenv()
# cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

# # Connect to GCP bucket
# client = storage.Client()
# bucket = client.bucket('domainome-data')
# blob = bucket.blob('uniprot_sprot.fasta')  # replace with your FASTA file name

# # Download FASTA as text
# fasta_text = blob.download_as_text()

# # List of UniProt IDs you want
# desired_ids = domainome_ids  # example, use a set for fast lookup

# # Dictionary to store sequences
# uniprot_dict = {}
# current_id = None
# current_seq = []

# # Parse FASTA line by line
# for line in fasta_text.splitlines():
#     if line.startswith('>'):
#         if current_id and current_id in desired_ids:
#             # Save previous sequence if in desired list
#             uniprot_dict[current_id] = ''.join(current_seq)
#         # Extract UniProt ID
#         uniprot_id = line.split('|')[1]  # standard UniProt header: >sp|P12345|PROT_HUMAN ...
#         if uniprot_id in desired_ids:
#             current_id = uniprot_id
#             current_seq = []
#         else:
#             current_id = None  # skip sequences we don't care about
#             current_seq = []
#     else:
#         if current_id:  # only accumulate if ID is in desired list
#             current_seq.append(line.strip())

# # Save the last sequence if it was desired
# if current_id and current_id in desired_ids:
#     uniprot_dict[current_id] = ''.join(current_seq)

# print(f"Retrieved {len(uniprot_dict)} sequences from FASTA")
# print(list(uniprot_dict.items())[:5])


Retrieved 428 sequences from FASTA
[('P00519', 'MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAARWNSKENLLAGPSENDPNLFVALYDFVASGDNTLSITKGEKLRVLGYNHNGEWCEAQTKNGQGWVPSNYITPVNSLEKHSWYHGPVSRNAAEYLLSSGINGSFLVRESESSPGQRSISLRYEGRVYHYRINTASDGKLYVSSESRFNTLAELVHHHSTVADGLITTLHYPAPKRNKPTVYGVSPNYDKWEMERTDITMKHKLGGGQYGEVYEGVWKKYSLTVAVKTLKEDTMEVEEFLKEAAVMKEIKHPNLVQLLGVCTREPPFYIITEFMTYGNLLDYLRECNRQEVNAVVLLYMATQISSAMEYLEKKNFIHRDLAARNCLVGENHLVKVADFGLSRLMTGDTYTAHAGAKFPIKWTAPESLAYNKFSIKSDVWAFGVLLWEIATYGMSPYPGIDLSQVYELLEKDYRMERPEGCPEKVYELMRACWQWNPSDRPSFAEIHQAFETMFQESSISDEVEKELGKQGVRGAVSTLLQAPELPTKTRTSRRAAEHRDTTDVPEMPHSKGQGESDPLDHEPAVSPLLPRKERGPPEGGLNEDERLLPKDKKTNLFSALIKKKKKTAPTPPKRSSSFREMDGQPERRGAGEEEGRDISNGALAFTPLDTADPAKSPKPSNGAGVPNGALRESGGSGFRSPHLWKKSSTLTSSRLATGEEEGGGSSSKRFLRSCSASCVPHGAKDTEWRSVTLPRDLQSTGRQFDSSTFGGHKSEKPALPRKRAGENRSDQVTRGTVTPPPRLVKKNEEAADEVFKDIMESSPGSSPPNLTPKPLRRQVTVAPASGLPHKEEAGKGSALGTPAAAEPVTPTSKAGSGAPGGTSKGPAEESRVRRHKHSSESPGRDKGKLSRLKPAPPPPPAASAGKAGGKPSQSPSQEAAGEAVLGAKTKATSLVDAVNSDAAKPSQPG

In [ ]:
# path = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/'
# with open(path+"dict_domainome_uniprot.pkl", "wb") as f:
#     pickle.dump(uniprot_dict, f)

In [10]:
def make_mutation_csv(domain_id):

    domain_id_list=domain_id.split("_")
    dom_pos = float(domain_id_list[-1])

    load_dotenv()
    cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

    client = storage.Client()
    bucket = client.bucket('domainome-data')
    blob = bucket.blob('SupplementaryTable2.txt')

    df = pd.read_csv(StringIO(blob.download_as_text()), sep='\t')

    df_one_protein = df.where(df['domain_ID'] == domain_id).dropna()

    dom_position = df_one_protein['position'] - dom_pos
    df_one_protein.insert(loc=0, column='real_position', value=dom_position)
    df_one_protein_ns = df_one_protein[df_one_protein['mut_aa'] != '*'].copy()

    df_one_protein_ns['wt_seq'] = df_one_protein_ns.apply(lambda row: 
                                                      insert_wt(row['aa_seq'], row['real_position'], row['wt_aa']),
                                                      axis=1)
    
    df_mutation = df_one_protein_ns[['wt_seq','real_position','mut_aa','normalized_fitness']]
    df_mutation
    df_mutation.to_csv("mutation_"+domain_id+".csv")

In [11]:
make_mutation_csv('A0A2R8Y422_PF00240_2')

In [3]:
load_dotenv()
cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

client = storage.Client()
bucket = client.bucket('domainome-data')
blob = bucket.blob('SupplementaryTable2.txt')

df = pd.read_csv(StringIO(blob.download_as_text()), sep='\t')

In [23]:
df.head()

,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
0,A0A2R8Y422_PF00240_2,A0A2R8Y422,*IFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,*,True,118.0,113.0,62.0,10.0,29.0,2.0,97.66667,0.030945,0.014885,-0.819050,0.208478,339
1,A0A2R8Y422_PF00240_2,A0A2R8Y422,AIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,A,False,219.0,277.0,217.0,86.0,225.0,137.0,237.66670,0.069376,0.006673,-0.280790,0.093461,339
2,A0A2R8Y422_PF00240_2,A0A2R8Y422,CIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,C,False,706.0,726.0,459.0,768.0,507.0,616.0,630.33330,0.082052,0.004141,-0.103250,0.057995,339
3,A0A2R8Y422_PF00240_2,A0A2R8Y422,DIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,D,False,407.0,431.0,323.0,508.0,159.0,111.0,387.00000,0.071003,0.005162,-0.258003,0.072296,339
4,A0A2R8Y422_PF00240_2,A0A2R8Y422,EIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,E,False,37.0,56.0,37.0,201.0,102.0,95.0,43.33333,0.116326,0.012085,0.376783,0.169263,339


Get data for one domain.  In this case Q8N1P7_PF00030_1358.  Translate position by initial domain position.

In [4]:
df_one_protein = df.where(df['domain_ID'] == 'Q8N1P7_PF00030_1358').dropna()

dom_position = df_one_protein['position'] - 1358.0
df_one_protein.insert(loc=0, column='real_position', value=dom_position)
df_one_protein_ns = df_one_protein[df_one_protein['mut_aa'] != '*'].copy()

df_one_protein_ns.head()

,real_position,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
433445,0.0,Q8N1P7_PF00030_1358,Q8N1P7,AQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETN...,I,1358.0,A,False,75.0,38.0,108.0,4.0,2.0,1.0,73.666667,0.001176,0.035376,-0.914642,0.168991,53.0
433446,0.0,Q8N1P7_PF00030_1358,Q8N1P7,CQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETN...,I,1358.0,C,False,57.0,30.0,50.0,2.0,5.0,0.0,45.666667,0.035269,0.037827,-0.751780,0.180697,53.0
433447,0.0,Q8N1P7_PF00030_1358,Q8N1P7,DQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETN...,I,1358.0,D,False,64.0,49.0,72.0,2.0,4.0,0.0,61.666670,0.016447,0.036889,-0.841690,0.176217,53.0
433448,0.0,Q8N1P7_PF00030_1358,Q8N1P7,EQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETN...,I,1358.0,E,False,121.0,98.0,62.0,3.0,6.0,0.0,93.666670,0.006830,0.029177,-0.887631,0.139376,53.0
433449,0.0,Q8N1P7_PF00030_1358,Q8N1P7,FQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETN...,I,1358.0,F,False,69.0,8.0,45.0,1.0,4.0,14.0,40.666670,0.067566,0.029007,-0.597499,0.138567,53.0


In [5]:
df_one_protein_ns['wt_seq'] = df_one_protein_ns.apply(lambda row: 
                                                      insert_wt(row['aa_seq'], row['real_position'], row['wt_aa']),
                                                      axis=1)

In [6]:
df_mutation = df_one_protein_ns[['wt_seq','real_position','mut_aa','normalized_fitness']]
df_mutation
df_mutation.to_csv("mutation_Q8N1P7.csv")

In [7]:
df_mutation.head()

,wt_seq,real_position,mut_aa,normalized_fitness
433445,IQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETN...,0.0,A,-0.914642
433446,IQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETN...,0.0,C,-0.751780
433447,IQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETN...,0.0,D,-0.841690
433448,IQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETN...,0.0,E,-0.887631
433449,IQLFSRPDFLGDHFSFEDDQAALPASFRPQSCRVHGGSWILFDETN...,0.0,F,-0.597499


Get data for one domain.  In this case P07316_PF00030_87.  Translate position by initial domain position.

In [4]:
df_one_protein = df.where(df['domain_ID'] == 'P07316_PF00030_87').dropna()

dom_position = df_one_protein['position'] - 87.0
df_one_protein.insert(loc=0, column='real_position', value=dom_position)
df_one_protein_ns = df_one_protein[df_one_protein['mut_aa'] != '*'].copy()

df_one_protein_ns.head()



,real_position,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
121785,0.0,P07316_PF00030_87,P07316,AAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,A,False,645.0,476.0,327.0,202.0,183.0,264.0,482.6667,0.067864,0.006483,0.163960,0.070917,385.0
121786,0.0,P07316_PF00030_87,P07316,CAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,C,False,1155.0,733.0,502.0,198.0,257.0,578.0,796.6667,0.061549,0.005723,0.094874,0.062609,385.0
121787,0.0,P07316_PF00030_87,P07316,DAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,D,False,1079.0,784.0,566.0,197.0,220.0,399.0,809.6667,0.054534,0.006000,0.018137,0.065632,385.0
121788,0.0,P07316_PF00030_87,P07316,EAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,E,False,305.0,251.0,187.0,75.0,94.0,182.0,247.6667,0.067474,0.008372,0.159687,0.091586,385.0
121789,0.0,P07316_PF00030_87,P07316,FAYRMKIYDRDELRGQMSELTDDCISVQDRFHLTEIHSLNVLEGSW...,G,87.0,F,False,803.0,628.0,463.0,241.0,412.0,729.0,631.3333,0.083318,0.004906,0.333017,0.053670,385.0


Insert mutation into correct position of protein sequence.

In [7]:
df_one_protein_ns['wt_seq'] = df_one_protein_ns.apply(lambda row: 
                                                      insert_wt(row['aa_seq'], row['real_position'], row['wt_aa']),
                                                      axis=1)


In [8]:
df_mutation = df_one_protein_ns[['wt_seq','real_position','mut_aa','normalized_fitness']]
df_mutation
df_mutation.to_csv("mutation.csv")

In [9]:
df_mutation = pd.read_csv('mutation.csv')

Initializing base model Pro Gen 2

In [24]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
device = 'cpu'
print(f"Using {device} device")

model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2_noeval(model_name)

Using cpu device


Initializing LoRA

In [7]:
for name, module in base_model.named_modules():
    print(name)


transformer
transformer.wte
transformer.drop
transformer.h
transformer.h.0
transformer.h.0.ln_1
transformer.h.0.attn
transformer.h.0.attn.attn_dropout
transformer.h.0.attn.resid_dropout
transformer.h.0.attn.qkv_proj
transformer.h.0.attn.out_proj
transformer.h.0.mlp
transformer.h.0.mlp.fc_in
transformer.h.0.mlp.fc_out
transformer.h.0.mlp.act
transformer.h.0.mlp.dropout
transformer.h.1
transformer.h.1.ln_1
transformer.h.1.attn
transformer.h.1.attn.attn_dropout
transformer.h.1.attn.resid_dropout
transformer.h.1.attn.qkv_proj
transformer.h.1.attn.out_proj
transformer.h.1.mlp
transformer.h.1.mlp.fc_in
transformer.h.1.mlp.fc_out
transformer.h.1.mlp.act
transformer.h.1.mlp.dropout
transformer.h.2
transformer.h.2.ln_1
transformer.h.2.attn
transformer.h.2.attn.attn_dropout
transformer.h.2.attn.resid_dropout
transformer.h.2.attn.qkv_proj
transformer.h.2.attn.out_proj
transformer.h.2.mlp
transformer.h.2.mlp.fc_in
transformer.h.2.mlp.fc_out
transformer.h.2.mlp.act
transformer.h.2.mlp.dropout
tran

In [25]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    # target_modules=["query", "key", "value", "output.dense"],
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
    # task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(base_model, lora_config)

In [26]:
for name, module in base_model.named_modules():
    if "qkv_proj" in name or "out_proj" in name:
        print(name, module)

transformer.h.0.attn.qkv_proj lora.Linear(
  (base_layer): Linear(in_features=1536, out_features=4608, bias=False)
  (lora_dropout): ModuleDict(
    (default): Dropout(p=0.1, inplace=False)
  )
  (lora_A): ModuleDict(
    (default): Linear(in_features=1536, out_features=8, bias=False)
  )
  (lora_B): ModuleDict(
    (default): Linear(in_features=8, out_features=4608, bias=False)
  )
  (lora_embedding_A): ParameterDict()
  (lora_embedding_B): ParameterDict()
  (lora_magnitude_vector): ModuleDict()
)
transformer.h.0.attn.qkv_proj.base_layer Linear(in_features=1536, out_features=4608, bias=False)
transformer.h.0.attn.qkv_proj.lora_dropout ModuleDict(
  (default): Dropout(p=0.1, inplace=False)
)
transformer.h.0.attn.qkv_proj.lora_dropout.default Dropout(p=0.1, inplace=False)
transformer.h.0.attn.qkv_proj.lora_A ModuleDict(
  (default): Linear(in_features=1536, out_features=8, bias=False)
)
transformer.h.0.attn.qkv_proj.lora_A.default Linear(in_features=1536, out_features=8, bias=False)
tra

Set device and optimizer

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
# optimizer = torch.optim.AdamW(
#     filter(lambda p: p.requires_grad, model.parameters()), lr=1e-2
# )

Load protein sequence and experimental data

In [28]:
protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
fitness_data = df_mutation

In [12]:
len(protein_seq)

90

Making experimental tensor

In [29]:
fitness_data.reset_index(drop=True, inplace=True)

positions = np.arange(len(protein_seq))
# amino_acids = ['L', 'A', 'G', 'V', 'S', 'E', 'R', 'T', 'I',
#                'D', 'P', 'K', 'Q', 'N', 'F', 'Y', 'M', 'H', 'W', 'C']

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]

Construct df2.  Mutation and real position in the domain.

In [30]:
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})

In [31]:
df2.tail()

,real_position,mut_aa
1795,89,S
1796,89,T
1797,89,V
1798,89,W
1799,89,Y


In [32]:
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')

Construct experimental tensor

In [18]:
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

Set up train, validate, test

In [33]:
all_indices = torch.randperm(seq_len)

num_train = int(seq_len * 0.5)
num_val   = int(seq_len * 0.25)
num_test  = seq_len - num_train - num_val

train_indices = all_indices[:num_train] # len 900
val_indices   = all_indices[num_train:num_train + num_val]
test_indices  = all_indices[num_train + num_val:]


In [44]:
torch.isnan(ft_tensor).sum().item()
len(ft_tensor)

1800

Fine-tuning model

In [46]:
train_losses = []
val_losses = []
early_stop_count = 0


vocab_dict = tokenizer.get_vocab()
seq_list = list(protein_seq)
inputs = tokenizer(protein_seq, return_tensors="pt").to(device)

for epoch in range(5):
    model.train()
    outputs = model(**inputs)
    logits = outputs.logits.squeeze(0)  # (seq_len, vocab_size)


    # Compute LLR ----
    # wt_logits = torch.log_softmax(logits, dim=-1)
    # wt_logits = wt_logits[1:-1, :]

    wt_logits = torch.log_softmax(logits, dim=-1)
    # wt_logits = wt_logits[1:-1, :]

    residue_indices = torch.arange(len(protein_seq))


    # ignoring BOS token
    seq_indices = [vocab_dict[aa] for aa in seq_list]
    wt_norm_tensor = wt_logits[residue_indices, seq_indices].unsqueeze(-1)
    LLR_tensor = wt_logits - wt_norm_tensor
    LLR_tensor_aa_only = LLR_tensor[:, aa_token_ids]

    #flatten the LLR_tensor
    flattened_LLR_tensor = LLR_tensor_aa_only.flatten()
    flattened_exp_tensor = exp_tensor.to(device)
    flattened_LLR_tensor = flattened_LLR_tensor.to(device)

    #to stack
    combined = torch.stack([flattened_LLR_tensor, flattened_exp_tensor], dim=0)
    ft_tensor = torch.transpose(combined, 0, 1)

    predicted_scores = []
    experimental_values = []

    train_tensor = ft_tensor[train_indices]

    #drop nan
    train_tensor = train_tensor[~torch.any(train_tensor.isnan(), dim=1)]

    num_samples = 45
    positions = train_tensor[torch.randperm(len(train_tensor))[:num_samples]]

    predicts = positions[:, 0] #LLR
    targets = positions[:, 1] #exp

# Compute loss with predicts and targets
    loss = listwise_ranking_loss(predicts, targets)
    # log_probs = torch.log_softmax(logits, dim=-1)
    # loss = -log_probs[torch.arange(len(seq_indices)), seq_indices].mean() # needs to be difference of predict/target
    loss.backward()
    # for name, param in model.named_parameters():
    #     if param.requires_grad:
    #         print(name, param.grad.abs().mean())
    optimizer.step()
    optimizer.zero_grad()

    train_losses.append(loss.item())

    # ----------- VALIDATION (no backprop) -----------
    model.eval()
    with torch.no_grad():
      val_tensor = ft_tensor[val_indices]
      val_tensor = val_tensor [~torch.any(val_tensor.isnan(), dim=1)]

      num_samples = 45
      positions = val_tensor[torch.randperm(len(val_tensor))[:num_samples]]

      predicts = positions[:, 0] #LLR
      targets = positions[:, 1] #exp

      val_loss = listwise_ranking_loss(predicts, targets)
    #   val_log_probs = torch.log_softmax(logits, dim=-1)
    #   val_loss = -log_probs[torch.arange(len(seq_indices)), seq_indices].mean()
      val_losses.append(val_loss.item())


    print(f"Epoch {epoch+1} - Training Loss: {loss.item():.4f} | Validation Loss: {val_loss.item():.4f}")

    if val_loss.item() > loss.item():
        early_stop_count += 1
    else:
        early_stop_count = 0

    # if early_stop_count > 2:
    #     print("Validation loss exceeded training loss 3 times — early stopping.")
    #     break
    print(early_stop_count)

Epoch 1 - Training Loss: 4.0722 | Validation Loss: 3.8892
0
Epoch 2 - Training Loss: 6.2683 | Validation Loss: 7.6440
1
Epoch 3 - Training Loss: 4.7224 | Validation Loss: 8.7543
2
Epoch 4 - Training Loss: 6.2425 | Validation Loss: 3.9620
0
Epoch 5 - Training Loss: 8.6166 | Validation Loss: 5.2189
0


In [46]:
#claude version

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

train_losses = []
val_losses = []
early_stop_count = 0
# patience = 3  # stop if validation loss worse than training for 3 epochs

vocab_dict = tokenizer.get_vocab()
seq_list = list(protein_seq)
inputs = tokenizer(protein_seq, return_tensors="pt").to(device)

for epoch in range(25):
    # -------------------- TRAINING --------------------
    model.train()
    outputs = model(**inputs)
    logits = outputs.logits.squeeze(0)  # (seq_len, vocab_size)

    # Compute log-softmax
    wt_logits = torch.log_softmax(logits, dim=-1)
    seq_indices = torch.tensor([vocab_dict[aa] for aa in seq_list], device=device)

    # Compute LLR
    wt_norm_tensor = wt_logits[torch.arange(len(seq_indices)), seq_indices].unsqueeze(-1)
    LLR_tensor = wt_logits - wt_norm_tensor
    LLR_tensor_aa_only = LLR_tensor[:, aa_token_ids]  # only relevant positions

    # Flatten LLR and stack with experimental values
    flattened_LLR = LLR_tensor_aa_only.flatten()
    flattened_exp = exp_tensor.to(device).flatten()
    combined = torch.stack([flattened_LLR, flattened_exp], dim=0).T  # shape (num_positions, 2)

    # ----- TRAINING SAMPLE -----
    train_tensor = combined[train_indices]
    train_tensor = train_tensor[~torch.any(torch.isnan(train_tensor), dim=1)]
    num_samples = min(20, len(train_tensor))  # sample at most 20
    positions = train_tensor[torch.randperm(len(train_tensor))[:num_samples]]

    predicts = positions[:, 0]  # LLR (model output)
    targets  = positions[:, 1]  # experimental values

    # Compute loss and update model
    # loss = listwise_ranking_loss(predicts, targets)
    log_probs = torch.log_softmax(logits, dim=-1)
    loss = -log_probs[torch.arange(len(seq_indices)), seq_indices].mean()

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    train_losses.append(loss.item())

    # -------------------- VALIDATION --------------------
    model.eval()
    with torch.no_grad():
        val_tensor = combined[val_indices]
        val_tensor = val_tensor[~torch.any(torch.isnan(val_tensor), dim=1)]
        num_samples = min(20, len(val_tensor))
        positions = val_tensor[torch.randperm(len(val_tensor))[:num_samples]]

        predicts = positions[:, 0]  # LLR (model output)
        targets  = positions[:, 1]  # experimental values

        val_loss = listwise_ranking_loss(predicts, targets)
        val_losses.append(val_loss.item())

    print(f"Epoch {epoch+1}: Train Loss={loss.item():.4f}, Val Loss={val_loss.item():.4f}")

    # -------------------- EARLY STOPPING --------------------
    if val_loss.item() > loss.item():
        early_stop_count += 1
    else:
        early_stop_count = 0

    # if early_stop_count >= patience:
        # print(f"Validation loss worse than training for {patience} epochs — stopping early.")
        # break


Epoch 1: Train Loss=2.8409, Val Loss=3.2123
Epoch 2: Train Loss=2.5069, Val Loss=3.5412
Epoch 3: Train Loss=2.7355, Val Loss=3.5857
Epoch 4: Train Loss=2.6267, Val Loss=2.9759
Epoch 5: Train Loss=2.5131, Val Loss=3.6371
Epoch 6: Train Loss=2.5017, Val Loss=3.2982
Epoch 7: Train Loss=2.5060, Val Loss=2.8247
Epoch 8: Train Loss=2.5434, Val Loss=5.1258
Epoch 9: Train Loss=2.5262, Val Loss=3.3827
Epoch 10: Train Loss=2.5601, Val Loss=3.5680
Epoch 11: Train Loss=2.5698, Val Loss=2.7012
Epoch 12: Train Loss=2.5365, Val Loss=2.8318
Epoch 13: Train Loss=2.5141, Val Loss=4.9416
Epoch 14: Train Loss=2.4272, Val Loss=3.6623
Epoch 15: Train Loss=2.3071, Val Loss=2.9301
Epoch 16: Train Loss=2.2520, Val Loss=3.8477
Epoch 17: Train Loss=2.2393, Val Loss=3.8944
Epoch 18: Train Loss=2.2191, Val Loss=2.9857
Epoch 19: Train Loss=2.2761, Val Loss=3.5107
Epoch 20: Train Loss=2.2045, Val Loss=4.0313
Epoch 21: Train Loss=2.2986, Val Loss=3.2254
Epoch 22: Train Loss=2.4294, Val Loss=2.8606
Epoch 23: Train Los

In [45]:
# chat gpt version
# Make sure model and base_model are on the same device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
base_model.to(device)

# Tokenize input sequence
inputs = tokenizer(protein_seq, return_tensors="pt").to(device)

# Compute logits for the base model (for comparison)
with torch.no_grad():
    out_base = base_model(**inputs).logits

# Make a single training step for PEFT model
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)  # small step

# Forward pass
outputs = model(**inputs)
logits = outputs.logits.squeeze(0)

# Example: compute simple loss to make gradients nonzero
# Here, let's just use negative log-likelihood for the sequence
vocab_dict = tokenizer.get_vocab()
seq_indices = torch.tensor([vocab_dict[aa] for aa in protein_seq], device=device)
log_probs = torch.log_softmax(logits, dim=-1)
loss = -log_probs[torch.arange(len(seq_indices)), seq_indices].mean()

# Backprop and one optimizer step
loss.backward()
optimizer.step()
optimizer.zero_grad()

# Compare outputs after one step
with torch.no_grad():
    out_peft = model(**inputs).logits
    diff = out_peft - out_base
    print("Max absolute difference:", diff.abs().max().item())
    print("First 10 tokens diff:", diff[0, :10])


Max absolute difference: 16.70229148864746
First 10 tokens diff: tensor([[ 2.3636e-01,  2.3524e-01, -2.4843e-02, -1.1305e-01,  6.3858e-02,
         -7.0484e-02,  4.4346e-01, -6.5460e-03,  2.5834e-01,  2.2706e-01,
         -1.2011e+00,  1.8514e-01,  2.7653e-02, -3.8088e-01, -2.3570e-01,
         -3.0223e-01, -2.4527e-01,  9.7426e-01,  7.4787e-02, -4.2507e-02,
          1.9923e-01, -3.2951e-02, -7.8927e-01, -1.4070e-01,  5.7520e-01,
         -1.9855e-01, -3.1161e-01, -8.5537e-02, -6.6688e-01,  2.2422e-01,
          2.3516e-01,  2.3354e-01],
        [ 2.4007e-01,  2.3892e-01, -2.0220e-02, -1.1382e-01,  5.5347e-02,
         -8.0799e-02,  4.5126e-01, -8.7547e-03,  2.5403e-01,  2.1624e-01,
         -1.2114e+00,  1.7251e-01,  2.6264e-02, -3.8932e-01, -2.4163e-01,
         -3.1277e-01, -2.4706e-01,  9.7871e-01,  7.9323e-02, -5.3619e-02,
          1.9191e-01, -4.7081e-02, -7.9730e-01, -1.4660e-01,  5.7718e-01,
         -2.0702e-01, -3.2552e-01, -9.1362e-02, -6.6901e-01,  2.2364e-01,
          2

In [20]:
type(model)

peft.peft_model.PeftModelForCausalLM

In [22]:
names = list(pg_dict.keys())

In [36]:
# name = 'PAI1_HUMAN'
name = 'S22A1_HUMAN'
pg_dict[name]
# pg_dict['S22A1_HUMAN']

{'sequence': 'PTVDDILEQVGESGWFQKQAFLILCLLSAAFAPICVGIVFLGFTPDHHCQSPGVAELSQRCGWSPAEELNYTVPGLGPAGEAFLGQCRRYEVDWNQSALSCVDPLASLATNRSHLPLGPCQDGWVYDTPGSSIVTEFNLVCADSWKLDLFQSCLNAGFLFGSLGVGYFADRFGRKLCLLGTVLVNAVSGVLMAFSPNYMSMLLFRLLQGLVSKGNWMAGYTLITEFVGSGSRRTVAIMYQMAFTVGLVALTGLAYALPHWRWLQLAVSLPTFLFLLYYWCVPESPRWLLSQKRNTEAIKIMDHIAQKNGKLPPADLKMLSLEEDVTEKLSPSFADLFRTPRLRKRTFILMYLWFTDSVLYQGLILHMGATSGNLYLDFLYSALVEIPGAFIALITIDRVGRIYPMAMSNLLAGAACLVMIFISPDLHWLNIIIMCVGRMGITIAIQMICLVNAELYPTFVRNLGVMVCSSLCDIGGIITPFIVFRLREVWQALPLILFAVLGLLAAGVTLLLPETKGVALPETMKDAENLGRKAKPKENTIYLKVQTSEPSGT',
 'DMS': array([[ 0.29803039,         nan, -0.02282262, ...,         nan,
                 nan, -0.40331238],
        [-0.0186989 , -0.9664416 ,  0.0705317 , ..., -0.81024862,
         -1.36953997,         nan],
        [-1.3560825 , -0.87940227, -2.38607073, ...,         nan,
         -0.20007489,  0.23612909],
        ...,
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
  

In [37]:
sequence = pg_dict[name]['sequence']

In [47]:
model.eval()

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]

input = tokenizer(sequence, return_tensors="pt").to(device)
outputs = model(**inputs)

prompt1 = "1"+sequence  # run it forwards
prompt2 = "2"+sequence[::-1]  # run it backwards

input_ids1 = torch.tensor(tokenizer.encode(prompt1)).unsqueeze(0).to(model.device)
with torch.no_grad():
    logits1 = model(input_ids1).logits
shift_logits1 = logits1[:, :-1, :]  # remove last entry

input_ids2 =  torch.tensor(tokenizer.encode(prompt2)).unsqueeze(0).to(model.device)
with torch.no_grad():
    logits2 = model(input_ids2).logits
shift_logits2 = logits2[:, :-1, :] # remove last entry

shift_logits2 = shift_logits2[:, torch.arange(shift_logits2.size(1) - 1, -1, -1), :]

input_ids = input_ids1[:, 1:]

# take averages of matrices, 2nd one in reverse order
# to simulate BERT output

logits = (shift_logits1 + shift_logits2)/2

log_probs = F.log_softmax(logits, dim = -1)
# n = log_probs.size(1)

ref_log_probs = log_probs[0, torch.arange(input_ids.size(1)), input_ids[0]]
ref_log_probs = ref_log_probs.unsqueeze(1)


llr_matrix = log_probs - ref_log_probs
llr_matrix = llr_matrix[0][:, aa_token_ids]
log_probs = log_probs[0][:, aa_token_ids]

In [48]:
log_probs

tensor([[-2.1215, -4.7629, -5.8157,  ..., -3.1387, -5.2349, -4.9685],
        [-2.4693, -4.5426, -3.9928,  ..., -4.3566, -7.0935, -5.6114],
        [-2.8378, -4.7669, -4.7057,  ..., -2.3763, -5.4062, -4.0159],
        ...,
        [-2.6842, -3.9872, -2.8762,  ..., -2.9067, -4.6968, -3.9412],
        [-2.6455, -4.0825, -2.8709,  ..., -2.8516, -4.7201, -3.9773],
        [-2.6717, -4.1247, -2.6987,  ..., -2.9773, -4.7063, -3.8908]])

In [39]:
pg_dict[name]['log_probs']

array([[-2.115878 , -4.763095 , -5.8257475, ..., -3.1382017, -5.240115 ,
        -4.972911 ],
       [-2.4704216, -4.5449915, -3.9992301, ..., -4.363641 , -7.107171 ,
        -5.6235895],
       [-2.8370202, -4.774276 , -4.7136984, ..., -2.3746636, -5.4226904,
        -4.024169 ],
       ...,
       [-2.682618 , -3.988488 , -2.876664 , ..., -2.9054878, -4.7014704,
        -3.9416206],
       [-2.6449132, -4.083321 , -2.870255 , ..., -2.851441 , -4.7234197,
        -3.9793477],
       [-2.6710882, -4.125266 , -2.6983404, ..., -2.9770117, -4.709655 ,
        -3.891654 ]], shape=(553, 20), dtype=float32)

In [54]:
for name, param in model.named_parameters():
    if 'lora' in name or 'adapter' in name:
        print(name, param.abs().max())

base_model.model.transformer.h.0.attn.qkv_proj.lora_A.default.weight tensor(0.4325, grad_fn=<MaxBackward1>)
base_model.model.transformer.h.0.attn.qkv_proj.lora_B.default.weight tensor(0.4395, grad_fn=<MaxBackward1>)
base_model.model.transformer.h.0.attn.out_proj.lora_A.default.weight tensor(0.4429, grad_fn=<MaxBackward1>)
base_model.model.transformer.h.0.attn.out_proj.lora_B.default.weight tensor(0.4115, grad_fn=<MaxBackward1>)
base_model.model.transformer.h.1.attn.qkv_proj.lora_A.default.weight tensor(0.3172, grad_fn=<MaxBackward1>)
base_model.model.transformer.h.1.attn.qkv_proj.lora_B.default.weight tensor(0.3424, grad_fn=<MaxBackward1>)
base_model.model.transformer.h.1.attn.out_proj.lora_A.default.weight tensor(0.3184, grad_fn=<MaxBackward1>)
base_model.model.transformer.h.1.attn.out_proj.lora_B.default.weight tensor(0.3570, grad_fn=<MaxBackward1>)
base_model.model.transformer.h.2.attn.qkv_proj.lora_A.default.weight tensor(0.3330, grad_fn=<MaxBackward1>)
base_model.model.transformer

In [29]:
model.print_trainable_parameters()

trainable params: 1,990,656 || all params: 766,794,272 || trainable%: 0.2596
